# Supp Table 4a: cell-type abundance per TIL zone

**Produces** two Supp Table 4a files:
- `til_zone_celltype_stats_long`: for each TIL group (TIL-high / TIL-low sections) and TIL zone,
  the mean log1p cell2location abundance (`means_cell_abundance_w_sf`) of each of the 72 level-4
  cell types over the zone's spots (`mean_in`) and over the other spots of the same sections
  (`mean_out`), their difference and log2 ratio (pseudocount 1e-6). Cell types absent from one
  centre's cell2location reference (Fibro_Cycling, cDC_CD207, T_other) are NaN, as published.
- `til_zone_celltype_summary`: per TIL group and zone, the 30 cell types (curated names) with the
  highest `mean_in` and the 30 with the highest log2 ratio.

TIL groups: 57 TIL-high / 171 TIL-low sections (`til.til_groups`); the 23 TIL-low sections without
TIL+ spots only have TIL− spots (148 sections in the other zones).

**Origin**: no code was found for these two tables (repo history on all branches, EFS
`Final_Notebooks`, `Analysis_Repo` incl. `TIL_Niche_Comparisons*.ipynb`, other notebooks on EFS);
the files were written on 2025-12-17 to EFS `results/visium/til_zones/` from the Visium object of
`TIL_Niche_Comparisons_v2_clean.ipynb` (Spencer Watson). This notebook was reconstructed from the
published columns: the statistic is the mean of log1p(abundance) (the raw mean, the median, per-section
means and the q05 abundances do not match), the log2 ratio uses a 1e-6 pseudocount, both in float32.

**Compared with the published tables**: see the last cell.

In [ ]:
import numpy as np
import pandas as pd

from gbmcollaboration import AllDataCenters
from gbmcollaboration.paper import annotation, config, output, til, visium

In [ ]:
OUT = output.setup("supptables_4a_til_zone_celltype_stats")
PUBLISHED_TABLES = config.EFS / "supplementary_tables"
GROUPS = ["TIL_high", "TIL_low"]
ZONES = ["TIL+", "BZ1", "BZ2", "TIL-"]
EPS = 1e-6
TOP_N = 30

In [ ]:
def zone_stats(abund, group, zone):
    """Mean log1p abundance inside / outside the zone, within the sections of one TIL group."""
    in_zone = ((spots["TIL_group"] == group) & (spots["TIL_zone"] == zone)).to_numpy()
    out_zone = ((spots["TIL_group"] == group) & (spots["TIL_zone"] != zone)).to_numpy()
    mean_in = np.log1p(abund[in_zone]).mean(axis=0)
    mean_out = np.log1p(abund[out_zone]).mean(axis=0)
    log2fc = np.log2((mean_in + np.float32(EPS)) / (mean_out + np.float32(EPS)))
    return pd.DataFrame({
        "TIL_group": group,
        "TIL_zone": zone,
        "n_spots_in": int(in_zone.sum()),
        "n_spots_out": int(out_zone.sum()),
        "n_samples_in": spots.loc[in_zone, "sample"].nunique(),
        "n_samples_out": spots.loc[out_zone, "sample"].nunique(),
        "celltype_level4": celltypes,
        "celltype_level4_curated": key.reindex(celltypes)["celltype_level4_curated"].to_numpy(),
        "mean_in": mean_in.astype(np.float64),
        "mean_out": mean_out.astype(np.float64),
        "delta_mean": (mean_in - mean_out).astype(np.float64),
        "log2fc_in_vs_out": log2fc.astype(np.float64),
    })


def top_names(df, col):
    return ", ".join(df.sort_values(col, ascending=False)["celltype_level4_curated"].head(TOP_N))

## Spots, TIL groups, cell2location abundances

In [ ]:
spots = til.til_zones()[["sample", "TIL_zone"]].copy()
spots["TIL_group"] = spots["sample"].map(til.til_groups()["TIL_group"])

c2l = AllDataCenters.load_cell2location(visium.sections()).drop(columns="total_cells")
c2l.index = [f"{s}_{b}" for s, b in c2l.index]
c2l = c2l.reindex(spots.index)
celltypes = c2l.columns.str.replace(visium.C2L_PREFIX, "", regex=False)
abund = c2l.to_numpy(dtype=np.float32)
key = annotation.celltype_key(efs_copy=True)
print(f"{len(spots):,} spots, {spots['sample'].nunique()} sections, {len(celltypes)} cell types; "
      f"cell types with missing values: {list(celltypes[np.isnan(abund).any(axis=0)])}")
print(spots.groupby("TIL_group")["sample"].nunique().to_string())

## Tables

In [ ]:
stats_long = pd.concat([zone_stats(abund, g, z) for g in GROUPS for z in ZONES], ignore_index=True)
summary = pd.DataFrame([
    {"TIL_group": g, "TIL_zone": z, "n_spots": d["n_spots_in"].iloc[0], "n_samples": d["n_samples_in"].iloc[0],
     "abundance_source_used": "c2l", "top_abundant": top_names(d, "mean_in"),
     "top_enriched_vs_rest_of_sample": top_names(d, "log2fc_in_vs_out")}
    for (g, z), d in stats_long.groupby(["TIL_group", "TIL_zone"], sort=False)
])
output.save_table(OUT, "til_zone_celltype_stats_long", stats_long,
                  panel="Supp Table 4a til_zone_celltype_stats_long", index=False)
output.save_table(OUT, "til_zone_celltype_summary", summary,
                  panel="Supp Table 4a til_zone_celltype_summary", index=False)
summary[["TIL_group", "TIL_zone", "n_spots", "n_samples"]]

## Comparison with the published tables

In [ ]:
pub = pd.read_csv(PUBLISHED_TABLES / "5a_til_zone_celltype_stats_long.csv")
new = pd.read_csv(OUT / "til_zone_celltype_stats_long.csv")
print(f"stats_long: shape published {pub.shape}, this run {new.shape}; "
      f"same columns {list(pub.columns) == list(new.columns)}")
text = [c for c in pub.columns if pub[c].dtype == object]
num = [c for c in pub.columns if c not in text]
print("  text columns identical:", (pub[text] == new[text]).all().all())
print("  NaN at the same places:", (pub[num].isna() == new[num].isna()).all().all())
print("  max abs difference:", (pub[num] - new[num]).abs().max().to_dict())

pub_s = pd.read_csv(PUBLISHED_TABLES / "5a_til_zone_celltype_summary.csv")
new_s = pd.read_csv(OUT / "til_zone_celltype_summary.csv")
print(f"summary: identical {pub_s.equals(new_s)}")
for name in ["til_zone_celltype_stats_long", "til_zone_celltype_summary"]:
    same = (OUT / f"{name}.csv").read_bytes() == (PUBLISHED_TABLES / f"5a_{name}.csv").read_bytes()
    print(f"{name}: byte-identical to the published file: {same}")

In [ ]:
print(output.versions(["numpy", "pandas", "anndata", "s3fs"]).to_string(index=False))